# Bloque 1: Fundamentos de Modelos y Mensajes

Para construir agentes autónomos con LangGraph, primero debemos entender la unidad más básica de LangChain: la comunicación directa con el Modelo de Lenguaje (LLM). 

En este notebook aprenderemos:
1. Cómo configurar el cliente del LLM.
2. Los 3 tipos de mensajes fundamentales (`System`, `Human`, `AI`).
3. La anatomía de una respuesta cuando el modelo decide usar una herramienta (Tool Calling).

In [ ]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno (Asegúrate de tener tu archivo .env configurado con OPENAI_API_KEY)
load_dotenv(dotenv_path="../.env")

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️ Error: No se encontró OPENAI_API_KEY. Revisa tu archivo .env")
else:
    print("✅ Entorno configurado correctamente.")

## 1. Chat Models y la Lista de Mensajes

En LangChain, no enviamos una simple cadena de texto al modelo. Le enviamos una **lista de mensajes** estructurada. Los principales son:
* `SystemMessage`: Define el comportamiento, contexto o "personalidad" del sistema (el prompt del sistema).
* `HumanMessage`: La entrada o petición del usuario.
* `AIMessage`: La respuesta generada por el modelo.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Instanciamos el modelo. 
# temperature=0 es vital para los agentes: queremos respuestas lógicas y deterministas, no creativas.
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# Construimos el historial/contexto
mensajes = [
    SystemMessage(content="Eres un asistente de base de datos corporativo. Responde de manera muy breve."),
    HumanMessage(content="¿Qué es una clave primaria en SQL?")
]

# Invocamos al modelo pasándole la lista
respuesta_normal = llm.invoke(mensajes)

print("--- Respuesta del Modelo ---")
print(f"Tipo de objeto: {type(respuesta_normal)}")
print(f"Contenido: {respuesta_normal.content}")

## 2. Preparando al Modelo para usar Herramientas

Cuando construyamos nuestro agente, el LLM no siempre responderá con texto. A veces, decidirá que necesita ejecutar una acción externa. 

Para que el modelo sepa qué acciones puede tomar, le "presentamos" las herramientas usando `.bind_tools()`. Observa cómo cambia la estructura de la respuesta (`AIMessage`) cuando el modelo decide usar una de ellas: el `.content` se queda vacío y se llena el atributo `.tool_calls`.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool # ¡Importante traer el decorador!

# 1. Definimos la función CON el decorador para que el esquema se genere perfecto
@tool
def buscar_cliente(nombre: str) -> str:
    """Busca la información de contacto de un cliente en la base de datos."""
    base_de_datos = {
        "Ana Lopez": "555-1010",
        "Carlos Ruiz": "555-2020",
        "Maria Gomez": "555-3030"
    }
    for cliente, telefono in base_de_datos.items():
        if nombre.lower() in cliente.lower():
            return f"{telefono}"
    return f"No se encontró a {nombre} en la base de datos."

# 2. Conectamos la herramienta al modelo
llm_con_herramientas = llm.bind_tools([buscar_cliente])

# 3. Hacemos el prompt de sistema más autoritario
mensajes_accion = [
    SystemMessage(content="Eres un asistente corporativo. DEBES usar la herramienta buscar_cliente para obtener los teléfonos que te pidan."),
    HumanMessage(content="¿Me puedes dar el teléfono de Ana?")
]

respuesta_accion = llm_con_herramientas.invoke(mensajes_accion)

print("--- 1. Decisión del Modelo ---")

if hasattr(respuesta_accion, "tool_calls") and respuesta_accion.tool_calls:
    print("¡Éxito! El modelo generó un 'tool_call':")
    tool_call = respuesta_accion.tool_calls[0]
    print(tool_call)

    # 4. Ejecutamos la herramienta 
    argumentos = tool_call["args"]
    resultado_herramienta = buscar_cliente.invoke(argumentos)

    print(f"\n--- 2. Ejecución de la Herramienta ---")
    print(f"Resultado real de Python: '{resultado_herramienta}'")

    # 5. Agregamos los mensajes al historial
    mensajes_accion.append(respuesta_accion)
    mensajes_accion.append(ToolMessage(
        content=str(resultado_herramienta), 
        tool_call_id=tool_call["id"],
        name=tool_call["name"]
    ))

    # 6. Segunda llamada para que el modelo lea el resultado
    respuesta_final = llm_con_herramientas.invoke(mensajes_accion)

    print("\n--- 3. Respuesta Final del Agente ---")
    print(respuesta_final.content)
    
else:
    print("❌ El modelo decidió NO usar la herramienta. Esta fue su respuesta en texto plano:")
    print(respuesta_accion.content)

## 3. El Agente Razonando: Manejo de Ambigüedad

Las herramientas no solo sirven para devolver datos exitosos. Pueden (y deben) devolver instrucciones para guiar el comportamiento del modelo cuando algo no sale como se esperaba.

En este ejemplo, le pediremos a la base de datos el teléfono de "Carlos". Como hay dos Carlos, la herramienta le devolverá al modelo un mensaje de "Ambigüedad", obligándolo a detenerse y preguntarnos el apellido.

In [ ]:
# 1. Definimos la herramienta con manejo de múltiples resultados
@tool
def buscar_telefono_cliente(nombre: str) -> str:
    """Busca la información de contacto de un cliente. Si hay ambigüedad, el LLM debe preguntar al usuario."""
    base_de_datos = {
        "Ana Lopez": "555-1010",
        "Carlos Ruiz": "555-2020",
        "Carlos Slim": "555-9999",
        "Maria Gomez": "555-3030"
    }
    
    # Guardamos todas las coincidencias encontradas
    coincidencias = []
    for cliente, telefono in base_de_datos.items():
        if nombre.lower() in cliente.lower():
            coincidencias.append(cliente)
            
    # Evaluamos los 3 escenarios posibles
    if len(coincidencias) == 0:
        return f"No se encontró a {nombre} en la base de datos."
    elif len(coincidencias) == 1:
        cliente_exacto = coincidencias[0]
        return f"{base_de_datos[cliente_exacto]}"
    else:
        return f"AMBIGÜEDAD: Se encontraron múltiples clientes: {', '.join(coincidencias)}."

# 2. Conectamos la herramienta
llm_con_herramientas = llm.bind_tools([buscar_telefono_cliente])

# 3. Hacemos la petición intencionalmente vaga
mensajes_accion = [
    SystemMessage(content="Eres un asistente corporativo. Usa tus herramientas. Si la herramienta reporta AMBIGÜEDAD, debes preguntarle al usuario para aclarar."),
    HumanMessage(content="¿Me das el teléfono de Carlos?")
]

print("Iniciando razonamiento...\n")

# --- PASO 1: El modelo decide llamar a la herramienta ---
respuesta_accion = llm_con_herramientas.invoke(mensajes_accion)
tool_call = respuesta_accion.tool_calls[0]
print(f"🤖 El modelo solicitó buscar: {tool_call['args']}")

# --- PASO 2: Ejecutamos el código Python ---
resultado_herramienta = buscar_telefono_cliente.invoke(tool_call["args"])
print(f"🔧 Resultado crudo de Python: '{resultado_herramienta}'\n")

# --- PASO 3: Le devolvemos el error al modelo ---
mensajes_accion.append(respuesta_accion)
mensajes_accion.append(ToolMessage(
    content=str(resultado_herramienta), 
    tool_call_id=tool_call["id"],
    name=tool_call["name"]
))

# --- PASO 4: El modelo evalúa el error y nos responde ---
respuesta_final = llm_con_herramientas.invoke(mensajes_accion)

print("--- Respuesta Final del Agente ---")
print(f"🤖 {respuesta_final.content}")

## 4. Cerrando el Ciclo: Resolviendo la Ambigüedad

Ahora que el agente nos preguntó a cuál "Carlos" nos referimos, vamos a responderle. 
Le pasaremos el nombre completo. El agente deberá leer el historial, darse cuenta de que ahora sí tiene el dato exacto, volver a llamar a la base de datos y darnos el teléfono.

In [ ]:
# --- PASO 5: Agregamos la pregunta del agente al historial ---
mensajes_accion.append(respuesta_final)

# --- PASO 6: El usuario responde aclarando el apellido ---
mensaje_aclaracion = HumanMessage(content="Ah, disculpa. Me refiero a Carlos Ruiz.")
mensajes_accion.append(mensaje_aclaracion)

print(f"👤 Usuario: {mensaje_aclaracion.content}\n")
print("Iniciando segundo razonamiento...\n")

# Volvemos a llamar al modelo que TIENE las herramientas
respuesta_aclaracion = llm_con_herramientas.invoke(mensajes_accion)

if hasattr(respuesta_aclaracion, "tool_calls") and respuesta_aclaracion.tool_calls:
    tool_call_2 = respuesta_aclaracion.tool_calls[0]
    print(f"🤖 El modelo solicitó buscar de nuevo: {tool_call_2['args']}")

    # Ejecutamos la herramienta con el nuevo argumento exacto
    resultado_2 = buscar_telefono_cliente.invoke(tool_call_2["args"])
    print(f"🔧 Resultado crudo de Python: '{resultado_2}'\n")

    # Devolvemos el resultado exitoso al modelo
    mensajes_accion.append(respuesta_aclaracion)
    mensajes_accion.append(ToolMessage(
        content=str(resultado_2), 
        tool_call_id=tool_call_2["id"],
        name=tool_call_2["name"]
    ))

    # Última llamada para la respuesta amigable
    respuesta_definitiva = llm_con_herramientas.invoke(mensajes_accion)

    # Limpiamos la salida por si el modelo devuelve una lista de bloques
    texto_final = respuesta_definitiva.content
    if isinstance(texto_final, list):
        texto_final = texto_final[0].get("text", str(texto_final))

    print("--- Respuesta Definitiva del Agente ---")
    print(f"🤖 {texto_final}")
    
else:
    print("❌ El modelo decidió responder sin buscar en la base de datos:")
    print(respuesta_aclaracion.content)

## 🎯 Conclusión del Bloque 1: ¿Por qué necesitamos LangGraph?

Si observas el código anterior, te darás cuenta de que tuvimos que hacer mucho trabajo manual:
1. Extraer el JSON del `tool_call`.
2. Ejecutar la función de Python a mano.
3. Hacer `.append()` del `AIMessage` y del `ToolMessage` para no perder el historial.
4. Volver a invocar al modelo, manejando condicionales `if/else` en cada paso.

Mantener este estado manualmente para un agente que usa 10 herramientas en un bucle de 20 pasos es insostenible y propenso a errores. 

**Aquí es donde entra LangGraph.** En el Bloque 3, veremos cómo un `StateGraph` automatiza todo este enrutamiento, ejecución y mantenimiento del historial (el `MessagesState`) con solo un par de nodos, permitiéndonos enfocarnos en la lógica de negocio y no en el *plumbing* de los mensajes.